In [ ]:
import numpy as np
import pandas as pd
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')
import itertools
import pyodbc
# from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error as smape
from dotenv.main import load_dotenv
import os

In [2]:
server = '10.3.4.139,1433'
database = 'dwh_prod'
username = 'developerptba'
password = 'LnfPYVFW1K2TAKf'

In [5]:
cnxn = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
cursor = cnxn.cursor()
query = "SELECT * FROM dwh.DM_financial_position_unpivot;"
df = pd.read_sql(query, cnxn)
df

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
0,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Actual,3.376864e+09,2023-01-31 05:11:01,2025-03-05 07:45:11.143
1,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,RKAP,2.840146e+09,2023-01-31 05:11:01,2025-03-05 07:45:11.143
2,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Optimize,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.143
3,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Moderate,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.143
4,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Worst,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.143
...,...,...,...,...,...,...,...,...,...,...,...,...
49407,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,RKAP,1.844921e+10,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49408,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,Optimize,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49409,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,Moderate,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49410,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,Worst,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177


In [ ]:
# df = pd.read_csv("D:/PT Bukit Asam/Document 2024/Script Python/DM_financial_position_unpivot_full.csv")

In [6]:
df

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
0,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Actual,3.376864e+09,2023-01-31 05:11:01,2025-03-05 07:45:11.143
1,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,RKAP,2.840146e+09,2023-01-31 05:11:01,2025-03-05 07:45:11.143
2,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Optimize,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.143
3,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Moderate,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.143
4,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,Worst,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.143
...,...,...,...,...,...,...,...,...,...,...,...,...
49407,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,RKAP,1.844921e+10,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49408,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,Optimize,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49409,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,Moderate,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49410,2024-08-31,8,2024,Operating Expenses,Depreciation and Amortization of Marketing,IDR,None,None,Worst,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177


In [7]:
df['BreakDown'].unique()

array(['Fuel Usage of Marketing', 'Other Selling Expenses of Marketing',
       'Cash Conversion Cycle', 'Salaries, Wages, and Employee Benefits',
       'Total Current Assets', 'Accured Expenses',
       'EBITDA After Minority', "Payment to stockholders' loan",
       'Cash from/(used in) Investing Activities', 'Cash Currency GBP',
       'Cash Currency USD', 'Payment from debt', 'USDIDR AVG',
       'Inventory Turnover', 'Others of Non Current Assets',
       'Non Current Portion of Interest Bearing Debt',
       'Receipt of tax refund', 'Revenue Derti', 'IPC Production',
       'Net Debt to EBITDA', 'Total Export Sales of Coal',
       'Balance Stock UPTE Live Stock', 'Business Travel and Education',
       'Dividen Receipt from Subsidiaries/Associate/JVCo.',
       'Ending Cash Balance', 'EBITDA Margin',
       'EBITDA to Interest Coverage Ratio', 'Operating Profit Margin',
       'Return on Equity',
       'Purchases of available-for-sale financial assets', 'NCI',
       'Coal Rea

In [8]:
df_SALES = df[df['BreakDown'].isin(['Total Domestic Sales of Coal','Total Export Sales of Coal'])].copy()
df_SALES

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
119,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,Actual,0.00,2023-01-31 05:11:01,2025-03-05 07:45:11.050
120,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,RKAP,0.00,2023-01-31 05:11:01,2025-03-05 07:45:11.050
121,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,Optimize,0.00,2023-01-31 05:11:01,2025-03-05 07:45:11.050
122,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,Moderate,0.00,2023-01-31 05:11:01,2025-03-05 07:45:11.050
123,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,Worst,0.00,2023-01-31 05:11:01,2025-03-05 07:45:11.050
...,...,...,...,...,...,...,...,...,...,...,...,...
49335,2024-08-31,8,2024,Total Export Sales,Total Export Sales of Coal,None,China,None,Amount_USD,10340280.91,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49336,2024-08-31,8,2024,Total Export Sales,Total Export Sales of Coal,None,China,None,Weight,144708.09,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49337,2024-08-31,8,2024,Total Export Sales,Total Export Sales of Coal,None,China,None,Load_TRH,77200.00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49338,2024-08-31,8,2024,Total Export Sales,Total Export Sales of Coal,None,China,None,Load_KPT,0.00,2024-08-31 05:11:01,2025-03-05 07:45:11.177


In [9]:
# df_overview_actual : jenis : aktual
df_SALES_actual = df_SALES[df_SALES['tipe'] == "Actual"]

# data diurutkan dari awal data
df_SALES_date = df_SALES_actual.sort_values(by='Date')

df_SALES_group = df_SALES_date.groupby(['Date'])['amount'].sum().reset_index()
df_SALES_group

,Date,amount
0,2022-01-31,2091660.10
1,2022-02-28,2056355.83
2,2022-03-31,2818667.63
3,2022-04-30,2728947.26
4,2022-05-31,2654839.01
5,2022-06-30,2273833.86
6,2022-07-31,2605831.36
7,2022-08-31,2963096.08
8,2022-09-30,3312228.82
9,2022-10-31,2960079.20


In [7]:
df_SALES_group.isna().sum()

Date      0
amount    0
dtype: int64

In [10]:
df_SALES_group.columns = ['ds','y']
df_SALES_group['ds'] = pd.to_datetime(df_SALES_group['ds'])

# Kombinasi parameter untuk tuning
changepoint_prior_scales = [0.01, 0.05,0.1]
seasonality_modes = ['additive']
seasonality_prior_scales = [0.01, 0.05, 1.0]

# Membuat list kombinasi parameter
param_combinations = list(itertools.product(changepoint_prior_scales, seasonality_modes, seasonality_prior_scales))

# Fungsi untuk menghitung SMAPE
def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(predicted) + np.abs(actual)))

# Menyimpan hasil tuning
best_smape = float('inf')
best_params = None

# Train dan test split
train_size = int(0.8 * len(df_SALES_group))
train_data = df_SALES_group[:train_size]
test_data = df_SALES_group[train_size:]

# Looping untuk setiap kombinasi parameter
for changepoint_prior_scale, seasonality_mode, seasonality_prior_scale in param_combinations:
    
    # Inisialisasi model Prophet dengan parameter tuning
    model = Prophet(
        yearly_seasonality=True, 
        weekly_seasonality=False, 
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_mode=seasonality_mode,
        seasonality_prior_scale=seasonality_prior_scale
    )
    model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    
    # Melatih model dengan data train
    model.fit(train_data)

    # Membuat dataframe masa depan yang mencakup periode data test
    future = model.make_future_dataframe(periods=len(test_data), freq='MS')

    # Melakukan prediksi
    forecast = model.predict(future)

    # Mengambil prediksi yang sesuai dengan data test
    forecast_test = forecast[-len(test_data):]

    # Menggabungkan data asli dengan prediksi
    test_data.loc[:, 'yhat'] = forecast_test['yhat'].values

    # Menghitung SMAPE untuk data test
    smape_value = smape(test_data['y'], test_data['yhat'])
    print(f"SMAPE dengan changepoint_prior_scale={changepoint_prior_scale}, seasonality_mode={seasonality_mode}, seasonality_prior_scale={seasonality_prior_scale}: {smape_value:.2f}%")

    # Menyimpan kombinasi parameter terbaik
    if smape_value < best_smape:
        best_smape = smape_value
        best_params = (changepoint_prior_scale, seasonality_mode, seasonality_prior_scale)

print(f"\nParameter terbaik: changepoint_prior_scale={best_params[0]}, seasonality_mode={best_params[1]}, seasonality_prior_scale={best_params[2]}")
print(f"SMAPE terbaik: {best_smape:.2f}%")

# Latih model dengan seluruh data menggunakan parameter terbaik
final_model = Prophet(
    yearly_seasonality=True, 
    weekly_seasonality=False, 
    daily_seasonality=False,
    changepoint_prior_scale=best_params[0],
    seasonality_mode=best_params[1],
    seasonality_prior_scale=best_params[2]
)
final_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)

# Melatih model dengan seluruh data
final_model.fit(df_SALES_group)

# Membuat dataframe masa depan untuk 6 bulan ke depan
future = final_model.make_future_dataframe(periods=12, freq='MS')

# Melakukan prediksi
forecast_final = final_model.predict(future)

# Menampilkan hasil prediksi
# print(forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

# Menyimpan hasil prediksi ke dalam DataFrame
forecast_df = forecast_final[['ds', 'yhat']]


07:46:08 - cmdstanpy - INFO - Chain [1] start processing
07:46:09 - cmdstanpy - INFO - Chain [1] done processing
07:46:09 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.01: 13.03%


07:46:09 - cmdstanpy - INFO - Chain [1] done processing
07:46:09 - cmdstanpy - INFO - Chain [1] start processing
07:46:10 - cmdstanpy - INFO - Chain [1] done processing


SMAPE dengan changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.05: 12.20%


07:46:10 - cmdstanpy - INFO - Chain [1] start processing


...

07:46:12 - cmdstanpy - INFO - Chain [1] done processing
07:46:12 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.05: 12.19%


07:46:32 - cmdstanpy - INFO - Chain [1] done processing
07:46:32 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=1.0: 87.97%

Parameter terbaik: changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.05
SMAPE terbaik: 12.19%


07:46:32 - cmdstanpy - INFO - Chain [1] done processing


In [11]:
forecast_df

,ds,yhat
0,2022-01-31,2.324474e+06
1,2022-02-28,2.389633e+06
2,2022-03-31,2.763748e+06
3,2022-04-30,2.656953e+06
4,2022-05-31,2.578058e+06
5,2022-06-30,2.604881e+06
6,2022-07-31,2.227871e+06
7,2022-08-31,2.867613e+06
8,2022-09-30,3.035381e+06
9,2022-10-31,3.022868e+06


In [12]:
# Menyusun Data Frame Sales
forecast_rows = pd.DataFrame({
    'Date': forecast_df['ds'],
    'Tonase': forecast_df['yhat'],
    'Jenis': 'Forecast',
    'Breakdown' : 'Total Sales'
})

df_actual_sales = pd.DataFrame({
    'Date': df_SALES_group['ds'],
    'Tonase': df_SALES_group['y'],
    'Jenis': 'Actual',
    'Breakdown' : 'Total Sales'
})

# Menggabungkan semua DataFrame
result_df_Sales = pd.concat([df_actual_sales,forecast_rows ], ignore_index=True)
result_df_Sales

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,2.091660e+06,Actual,Total Sales
1,2022-02-28,2.056356e+06,Actual,Total Sales
2,2022-03-31,2.818668e+06,Actual,Total Sales
3,2022-04-30,2.728947e+06,Actual,Total Sales
4,2022-05-31,2.654839e+06,Actual,Total Sales
...,...,...,...,...
81,2025-09-01,3.909972e+06,Forecast,Total Sales
82,2025-10-01,4.079944e+06,Forecast,Total Sales
83,2025-11-01,4.098804e+06,Forecast,Total Sales
84,2025-12-01,3.868628e+06,Forecast,Total Sales


In [13]:
df_Sales_Ekspor = df_SALES_actual[df_SALES_actual['BreakDown']=='Total Export Sales of Coal']
df_Sales_Ekspor

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
119,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,Actual,0.0,2023-01-31 05:11:01,2025-03-05 07:45:11.050
142,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,Turkey,None,Actual,0.0,2023-01-31 05:11:01,2025-03-05 07:45:11.050
185,2024-04-30,4,2024,Total Export Sales,Total Export Sales of Coal,None,Bangladesh,None,Actual,0.0,2024-04-30 05:11:01,2025-03-05 07:45:11.177
262,2023-01-31,1,2023,Total Export Sales,Total Export Sales of Coal,None,England,None,Actual,0.0,2023-01-31 05:11:01,2025-03-05 07:45:11.050
272,2024-04-30,4,2024,Total Export Sales,Total Export Sales of Coal,None,Cambodia,None,Actual,0.0,2024-04-30 05:11:01,2025-03-05 07:45:11.177
...,...,...,...,...,...,...,...,...,...,...,...,...
49145,2024-07-31,7,2024,Total Export Sales,Total Export Sales of Coal,None,Cambodia,None,Actual,0.0,2024-07-31 05:11:01,2025-03-05 07:45:11.177
49175,2023-12-31,12,2023,Total Export Sales,Total Export Sales of Coal,None,Myanmar,None,Actual,0.0,2023-12-31 05:11:01,2025-03-05 07:45:11.160
49199,2023-12-31,12,2023,Total Export Sales,Total Export Sales of Coal,None,Singapore,None,Actual,0.0,2023-12-31 05:11:01,2025-03-05 07:45:11.160
49293,2024-07-31,7,2024,Total Export Sales,Total Export Sales of Coal,None,Sri Lanka,None,Actual,0.0,2024-07-31 05:11:01,2025-03-05 07:45:11.177


In [14]:
df_Sales_Ekspor = df_Sales_Ekspor.sort_values(by='Date')

df_Sales_Ekspor = df_Sales_Ekspor.groupby(['Date'])['amount'].sum().reset_index()
df_Sales_Ekspor = df_Sales_Ekspor[df_Sales_Ekspor['amount']!= 0]
df_Sales_Ekspor

,Date,amount
0,2022-01-31,344709.49
1,2022-02-28,590855.99
2,2022-03-31,1350178.55
3,2022-04-30,1027221.48
4,2022-05-31,1025365.98
5,2022-06-30,824570.98
6,2022-07-31,934896.26
7,2022-08-31,1279021.74
8,2022-09-30,1664507.92
9,2022-10-31,1499129.00


In [15]:
df_Sales_Ekspor.columns = ['ds','y']
df_Sales_Ekspor['ds'] = pd.to_datetime(df_Sales_Ekspor['ds'])

# Kombinasi parameter untuk tuning
changepoint_prior_scales = [0.01, 0.05,0.1]
seasonality_modes = ['additive']
seasonality_prior_scales = [0.01, 0.05, 1.0]

# Membuat list kombinasi parameter
param_combinations = list(itertools.product(changepoint_prior_scales, seasonality_modes, seasonality_prior_scales))

# Fungsi untuk menghitung SMAPE
def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(predicted) + np.abs(actual)))

# Menyimpan hasil tuning
best_smape = float('inf')
best_params = None

# Train dan test split
train_size = int(0.8 * len(df_Sales_Ekspor))
train_data = df_Sales_Ekspor[:train_size]
test_data = df_Sales_Ekspor[train_size:]

# Looping untuk setiap kombinasi parameter
for changepoint_prior_scale, seasonality_mode, seasonality_prior_scale in param_combinations:
    
    # Inisialisasi model Prophet dengan parameter tuning
    model = Prophet(
        yearly_seasonality=True, 
        weekly_seasonality=True, 
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_mode=seasonality_mode,
        seasonality_prior_scale=seasonality_prior_scale,
        growth='linear'
    )
    model.add_seasonality(name='monthly', period=30, fourier_order=3)
    
    # Melatih model dengan data train
    model.fit(train_data)

    # Membuat dataframe masa depan yang mencakup periode data test
    future = model.make_future_dataframe(periods=len(test_data), freq='MS')

    # Melakukan prediksi
    forecast = model.predict(future)

    # Mengambil prediksi yang sesuai dengan data test
    forecast_test = forecast[-len(test_data):]

    # Menggabungkan data asli dengan prediksi
    test_data.loc[:, 'yhat'] = forecast_test['yhat'].values

    # Menghitung SMAPE untuk data test
    smape_value = smape(test_data['y'], test_data['yhat'])
    print(f"SMAPE dengan changepoint_prior_scale={changepoint_prior_scale}, seasonality_mode={seasonality_mode}, seasonality_prior_scale={seasonality_prior_scale}: {smape_value:.2f}%")

    # Menyimpan kombinasi parameter terbaik
    if smape_value < best_smape:
        best_smape = smape_value
        best_params = (changepoint_prior_scale, seasonality_mode, seasonality_prior_scale)

print(f"\nParameter terbaik: changepoint_prior_scale={best_params[0]}, seasonality_mode={best_params[1]}, seasonality_prior_scale={best_params[2]}")
print(f"SMAPE terbaik: {best_smape:.2f}%")

# Latih model dengan seluruh data menggunakan parameter terbaik
final_model = Prophet(
    yearly_seasonality=True, 
    weekly_seasonality=True, 
    daily_seasonality=False,
    changepoint_prior_scale=best_params[0],
    seasonality_mode=best_params[1],
    seasonality_prior_scale=best_params[2],
    growth='linear'
)
final_model.add_seasonality(name='monthly', period=30, fourier_order=3)

# Melatih model dengan seluruh data
final_model.fit(df_Sales_Ekspor)

# Membuat dataframe masa depan untuk 6 bulan ke depan
future = final_model.make_future_dataframe(periods=12, freq='MS')

# Melakukan prediksi
forecast_final = final_model.predict(future)

# Menampilkan hasil prediksi
# print(forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

# Menyimpan hasil prediksi ke dalam DataFrame
forecast_df_sales_ekspor = forecast_final[['ds', 'yhat']]

07:46:51 - cmdstanpy - INFO - Chain [1] start processing
07:46:52 - cmdstanpy - INFO - Chain [1] done processing
07:46:52 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.01: 20.73%


07:46:52 - cmdstanpy - INFO - Chain [1] done processing
07:46:52 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.05: 26.20%


07:46:54 - cmdstanpy - INFO - Chain [1] done processing
07:46:54 - cmdstanpy - INFO - Chain [1] start processing


...

07:46:59 - cmdstanpy - INFO - Chain [1] done processing
07:46:59 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.05: 26.00%


07:47:21 - cmdstanpy - INFO - Chain [1] done processing
07:47:21 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=1.0: 105.45%

Parameter terbaik: changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.01
SMAPE terbaik: 20.39%


07:47:21 - cmdstanpy - INFO - Chain [1] done processing


In [16]:
# # Menyimpan hasil prediksi ke dalam DataFrame
# forecast_df_sales_ekspor = forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
# forecast_df_sales_ekspor.columns = ['Date', 'Forecast', 'Lower Bound', 'Upper Bound']

# # Membulatkan kolom Forecast, Lower Bound, dan Upper Bound ke dua angka di belakang koma
# forecast_df_sales_ekspor['Forecast'] = forecast_df_sales_ekspor['Forecast'].round(2)
# forecast_df_sales_ekspor['Lower Bound'] = forecast_df_sales_ekspor['Lower Bound'].round(2)
# forecast_df_sales_ekspor['Upper Bound'] = forecast_df_sales_ekspor['Upper Bound'].round(2)
# # --- Tambahan Akumulasi Forecast per Tahun ---
# # Menambahkan kolom tahun berdasarkan kolom 'Date'

# forecast_df_sales_ekspor['year'] = forecast_df_sales_ekspor['Date'].dt.year

# # Menghitung akumulasi kumulatif per tahun
# cumulative_forecast_yearly = forecast_df_sales_ekspor.groupby('year')[['Forecast', 'Lower Bound', 'Upper Bound']].cumsum()

# # Menyimpan hasil kumulatif ke dalam DataFrame baru
# cumulative_forecast_df_sales_ekspor_yearly = pd.DataFrame({
#     'Date': forecast_df_sales_ekspor['Date'],
#     'Year': forecast_df_sales_ekspor['year'],
#     'Cumulative Forecast': cumulative_forecast_yearly['Forecast'].round(2),
#     'Cumulative Lower Bound': cumulative_forecast_yearly['Lower Bound'].round(2),
#     'Cumulative Upper Bound': cumulative_forecast_yearly['Upper Bound'].round(2)
# })

# # Menampilkan DataFrame kumulatif hasil prediksi per tahun
# # print("\nData kumulatif forecast per tahun:")
# # print(cumulative_forecast_df_sales_ekspor_yearly)

# # Pastikan kolom 'Date' dalam format datetime
# forecast_df_sales_ekspor['Date'] = pd.to_datetime(forecast_df_sales_ekspor['Date'])
# Menambahkan data untuk 'Forecast'
forecast_rows_SE = pd.DataFrame({
    'Date': forecast_df_sales_ekspor['ds'],
    'Tonase': forecast_df_sales_ekspor['yhat'],
    'Jenis': 'Forecast',
    'Breakdown' : 'Total Sales Ekspor'
})

df_actual_sales_ekspor = pd.DataFrame({
    'Date': df_Sales_Ekspor['ds'],
    'Tonase': df_Sales_Ekspor['y'],
    'Jenis': 'Actual',
    'Breakdown' : 'Total Sales Ekspor'
})

# cumulative_forecast_rows_ekspor = pd.DataFrame({
#     'ds': cumulative_forecast_df_sales_ekspor_yearly['Date'],
#     'y': cumulative_forecast_df_sales_ekspor_yearly['Cumulative Forecast'],
#     'jenis': 'cumulative forecast',
#     'Breakdown': 'Total Sales Ekspor'
# })

# Menggabungkan semua DataFrame
result_df_Sales_Export = pd.concat([df_actual_sales_ekspor,forecast_rows_SE ], ignore_index=True)
result_df_Sales_Export

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,3.447095e+05,Actual,Total Sales Ekspor
1,2022-02-28,5.908560e+05,Actual,Total Sales Ekspor
2,2022-03-31,1.350179e+06,Actual,Total Sales Ekspor
3,2022-04-30,1.027221e+06,Actual,Total Sales Ekspor
4,2022-05-31,1.025366e+06,Actual,Total Sales Ekspor
...,...,...,...,...
79,2025-09-01,2.013334e+06,Forecast,Total Sales Ekspor
80,2025-10-01,2.044098e+06,Forecast,Total Sales Ekspor
81,2025-11-01,2.100759e+06,Forecast,Total Sales Ekspor
82,2025-12-01,2.075156e+06,Forecast,Total Sales Ekspor


-----

In [17]:
df_Sales_Domestic = df_SALES_actual[df_SALES_actual['BreakDown']=='Total Domestic Sales of Coal']
df_Sales_Domestic

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
719,2023-01-31,1,2023,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1748073.90,2023-01-31 05:11:01,2025-03-05 07:45:11.050
1684,2024-04-30,4,2024,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,2077174.33,2024-04-30 05:11:01,2025-03-05 07:45:11.177
3600,2022-05-31,5,2022,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1629473.03,2022-05-31 05:11:01,2025-03-05 07:45:11.083
3953,2024-02-29,2,2024,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1678156.59,2024-02-29 05:11:01,2025-03-05 07:45:11.160
9032,2024-01-31,1,2024,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,2275939.28,2024-01-31 05:11:01,2025-03-05 07:45:11.160
9815,2023-03-31,3,2023,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1655423.69,2023-03-31 05:11:01,2025-03-05 07:45:11.130
10956,2023-02-28,2,2023,Total Domestic Sales,Total Domestic Sales of Coal,Ton,None,None,Actual,1753328.56,2023-02-28 05:11:01,2025-03-05 07:45:11.130
12234,2024-03-31,3,2024,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1902777.33,2024-03-31 05:11:01,2025-03-05 07:45:11.177
12730,2022-04-30,4,2022,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1701725.78,2022-04-30 05:11:01,2025-03-05 07:45:11.083
13355,2022-07-31,7,2022,Total Domestic Sales,Total Domestic Sales of Coal,None,None,None,Actual,1670935.10,2022-07-31 05:11:01,2025-03-05 07:45:11.097


In [18]:
df_Sales_Domestic = df_Sales_Domestic.sort_values(by='Date')

df_Sales_Domestic = df_Sales_Domestic.groupby(['Date'])['amount'].sum().reset_index()
df_Sales_Domestic

,Date,amount
0,2022-01-31,1746950.61
1,2022-02-28,1465499.84
2,2022-03-31,1468489.08
3,2022-04-30,1701725.78
4,2022-05-31,1629473.03
5,2022-06-30,1449262.88
6,2022-07-31,1670935.10
7,2022-08-31,1684074.34
8,2022-09-30,1647720.90
9,2022-10-31,1460950.20


In [19]:
df_Sales_Domestic.columns = ['ds','y']
df_Sales_Domestic['ds'] = pd.to_datetime(df_Sales_Domestic['ds'])

# Kombinasi parameter untuk tuning
changepoint_prior_scales = [ 0.1, 0.5, 1]
seasonality_modes = ['additive']
seasonality_prior_scales = [0.01, 0.1]

# Membuat list kombinasi parameter
param_combinations = list(itertools.product(changepoint_prior_scales, seasonality_modes, seasonality_prior_scales))

# Fungsi untuk menghitung SMAPE
def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(predicted) + np.abs(actual)))

# Menyimpan hasil tuning
best_smape = float('inf')
best_params = None

# Train dan test split
train_size = int(0.8 * len(df_Sales_Domestic))
train_data = df_Sales_Domestic[:train_size]
test_data = df_Sales_Domestic[train_size:]

# Looping untuk setiap kombinasi parameter
for changepoint_prior_scale, seasonality_mode, seasonality_prior_scale in param_combinations:
    
    # Inisialisasi model Prophet dengan parameter tuning
    model = Prophet(
        yearly_seasonality=True, 
        weekly_seasonality=True, 
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_mode=seasonality_mode,
        seasonality_prior_scale=seasonality_prior_scale,
        growth='linear'
    )
    model.add_seasonality(name='monthly', period=30, fourier_order=3)
    
    # Melatih model dengan data train
    model.fit(train_data)

    # Membuat dataframe masa depan yang mencakup periode data test
    future = model.make_future_dataframe(periods=len(test_data), freq='MS')

    # Melakukan prediksi
    forecast = model.predict(future)

    # Mengambil prediksi yang sesuai dengan data test
    forecast_test = forecast[-len(test_data):]

    # Menggabungkan data asli dengan prediksi
    test_data.loc[:, 'yhat'] = forecast_test['yhat'].values

    # Menghitung SMAPE untuk data test
    smape_value = smape(test_data['y'], test_data['yhat'])
    print(f"SMAPE dengan changepoint_prior_scale={changepoint_prior_scale}, seasonality_mode={seasonality_mode}, seasonality_prior_scale={seasonality_prior_scale}: {smape_value:.2f}%")

    # Menyimpan kombinasi parameter terbaik
    if smape_value < best_smape:
        best_smape = smape_value
        best_params = (changepoint_prior_scale, seasonality_mode, seasonality_prior_scale)

print(f"\nParameter terbaik: changepoint_prior_scale={best_params[0]}, seasonality_mode={best_params[1]}, seasonality_prior_scale={best_params[2]}")
print(f"SMAPE terbaik: {best_smape:.2f}%")

# Latih model dengan seluruh data menggunakan parameter terbaik
final_model = Prophet(
    yearly_seasonality=True, 
    weekly_seasonality=True, 
    daily_seasonality=False,
    changepoint_prior_scale=best_params[0],
    seasonality_mode=best_params[1],
    seasonality_prior_scale=best_params[2],
    growth='linear'
)
final_model.add_seasonality(name='monthly', period=30, fourier_order=3)

# Melatih model dengan seluruh data
final_model.fit(df_Sales_Domestic)

# Membuat dataframe masa depan untuk 6 bulan ke depan
future = final_model.make_future_dataframe(periods=12, freq='MS')

# Melakukan prediksi
forecast_final = final_model.predict(future)

# Menampilkan hasil prediksi
# print(forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

# Menyimpan hasil prediksi ke dalam DataFrame
forecast_df_sales_df_Sales_Domestic = forecast_final[['ds', 'yhat']]


07:52:17 - cmdstanpy - INFO - Chain [1] start processing
07:52:17 - cmdstanpy - INFO - Chain [1] done processing
07:52:18 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.01: 7.76%


07:52:18 - cmdstanpy - INFO - Chain [1] done processing
07:52:18 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.1: 11.16%


07:52:19 - cmdstanpy - INFO - Chain [1] done processing
07:52:19 - cmdstanpy - INFO - Chain [1] start processing


...

07:52:39 - cmdstanpy - INFO - Chain [1] done processing
07:52:39 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.01: 4.24%


07:52:58 - cmdstanpy - INFO - Chain [1] done processing
07:52:58 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.1: 90.97%

Parameter terbaik: changepoint_prior_scale=0.5, seasonality_mode=additive, seasonality_prior_scale=0.01
SMAPE terbaik: 3.87%


07:52:59 - cmdstanpy - INFO - Chain [1] done processing


In [20]:

# Menambahkan data untuk 'Forecast'
forecast_rows_SE = pd.DataFrame({
    'Date': forecast_df_sales_df_Sales_Domestic['ds'],
    'Tonase': forecast_df_sales_df_Sales_Domestic['yhat'],
    'Jenis': 'Forecast',
    'Breakdown' : 'Total Sales Domestik'
})

df_actual_sales_forecast_df_sales_df_Sales_Domestic = pd.DataFrame({
    'Date': df_Sales_Domestic['ds'],
    'Tonase': df_Sales_Domestic['y'],
    'Jenis': 'Actual',
    'Breakdown' : 'Total Sales Domestik'
})


# Menggabungkan semua DataFrame
result_df_SD = pd.concat([df_actual_sales_forecast_df_sales_df_Sales_Domestic,forecast_rows_SE], ignore_index=True)
result_df_SD

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,1.746951e+06,Actual,Total Sales Domestik
1,2022-02-28,1.465500e+06,Actual,Total Sales Domestik
2,2022-03-31,1.468489e+06,Actual,Total Sales Domestik
3,2022-04-30,1.701726e+06,Actual,Total Sales Domestik
4,2022-05-31,1.629473e+06,Actual,Total Sales Domestik
...,...,...,...,...
81,2025-09-01,1.613172e+06,Forecast,Total Sales Domestik
82,2025-10-01,1.671364e+06,Forecast,Total Sales Domestik
83,2025-11-01,1.627764e+06,Forecast,Total Sales Domestik
84,2025-12-01,1.563317e+06,Forecast,Total Sales Domestik


----

In [21]:
df_ASP = df[df['BreakDown']=='Average Market Price IDR']
df_ASP =df_ASP[df_ASP['tipe']=='Actual']


In [22]:
df_ASP

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
1454,2023-01-31,1,2023,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,1230003.11,2023-01-31 05:11:01,2025-03-05 07:45:11.050
1541,2024-04-30,4,2024,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,952847.66,2024-04-30 05:11:01,2025-03-05 07:45:11.177
2165,2024-02-29,2,2024,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,967697.62,2024-02-29 05:11:01,2025-03-05 07:45:11.160
5571,2022-01-31,1,2022,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,930067.30,2022-01-31 05:11:01,2025-03-05 07:45:11.020
5875,2022-03-31,3,2022,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,1412002.98,2022-03-31 05:11:01,2025-03-05 07:45:11.083
6323,2023-03-31,3,2023,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,1094070.92,2023-03-31 05:11:01,2025-03-05 07:45:11.130
9128,2024-01-31,1,2024,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,943557.59,2024-01-31 05:11:01,2025-03-05 07:45:11.160
9413,2023-04-30,4,2023,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,1074880.03,2023-04-30 05:11:01,2025-03-05 07:45:11.130
10385,2022-04-30,4,2022,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,1406561.92,2022-04-30 05:11:01,2025-03-05 07:45:11.083
10679,2022-05-31,5,2022,External Data,Average Market Price IDR,IDR/Ton,None,None,Actual,1279439.86,2022-05-31 05:11:01,2025-03-05 07:45:11.083


In [23]:
df_ASP = df_ASP.sort_values(by='Date')

df_ASP = df_ASP.groupby(['Date'])['amount'].sum().reset_index()
df_ASP

,Date,amount
0,2022-01-31,930067.30
1,2022-02-28,1046655.14
2,2022-03-31,1412002.98
3,2022-04-30,1406561.92
4,2022-05-31,1279439.86
5,2022-06-30,1260877.38
6,2022-07-31,1436669.88
7,2022-08-31,1473301.70
8,2022-09-30,1323887.35
9,2022-10-31,1619053.21


In [24]:
df_ASP.columns = ['ds','y']
df_ASP['ds'] = pd.to_datetime(df_ASP['ds'])

# Kombinasi parameter untuk tuning
changepoint_prior_scales = [ 0.1, 0.5, 1]
seasonality_modes = ['additive']
seasonality_prior_scales = [0.01, 0.1]

# Membuat list kombinasi parameter
param_combinations = list(itertools.product(changepoint_prior_scales, seasonality_modes, seasonality_prior_scales))

# Fungsi untuk menghitung SMAPE
def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(predicted) + np.abs(actual)))

# Menyimpan hasil tuning
best_smape = float('inf')
best_params = None

# Train dan test split
train_size = int(0.8 * len(df_ASP))
train_data = df_ASP[:train_size]
test_data = df_ASP[train_size:]

# Looping untuk setiap kombinasi parameter
for changepoint_prior_scale, seasonality_mode, seasonality_prior_scale in param_combinations:
    
    # Inisialisasi model Prophet dengan parameter tuning
    model = Prophet(
        yearly_seasonality=True, 
        weekly_seasonality=True, 
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_mode=seasonality_mode,
        seasonality_prior_scale=seasonality_prior_scale,
        growth='linear'
    )
    model.add_seasonality(name='monthly', period=30, fourier_order=3)
    
    # Melatih model dengan data train
    model.fit(train_data)

    # Membuat dataframe masa depan yang mencakup periode data test
    future = model.make_future_dataframe(periods=len(test_data), freq='MS')

    # Melakukan prediksi
    forecast = model.predict(future)

    # Mengambil prediksi yang sesuai dengan data test
    forecast_test = forecast[-len(test_data):]

    # Menggabungkan data asli dengan prediksi
    test_data.loc[:, 'yhat'] = forecast_test['yhat'].values

    # Menghitung SMAPE untuk data test
    smape_value = smape(test_data['y'], test_data['yhat'])
    print(f"SMAPE dengan changepoint_prior_scale={changepoint_prior_scale}, seasonality_mode={seasonality_mode}, seasonality_prior_scale={seasonality_prior_scale}: {smape_value:.2f}%")

    # Menyimpan kombinasi parameter terbaik
    if smape_value < best_smape:
        best_smape = smape_value
        best_params = (changepoint_prior_scale, seasonality_mode, seasonality_prior_scale)

print(f"\nParameter terbaik: changepoint_prior_scale={best_params[0]}, seasonality_mode={best_params[1]}, seasonality_prior_scale={best_params[2]}")
print(f"SMAPE terbaik: {best_smape:.2f}%")

# Latih model dengan seluruh data menggunakan parameter terbaik
final_model = Prophet(
    yearly_seasonality=True, 
    weekly_seasonality=True, 
    daily_seasonality=False,
    changepoint_prior_scale=best_params[0],
    seasonality_mode=best_params[1],
    seasonality_prior_scale=best_params[2],
    growth='linear'
)
final_model.add_seasonality(name='monthly', period=30, fourier_order=3)

# Melatih model dengan seluruh data
final_model.fit(df_ASP)

# Membuat dataframe masa depan untuk 6 bulan ke depan
future = final_model.make_future_dataframe(periods=12, freq='MS')

# Melakukan prediksi
forecast_final = final_model.predict(future)

# Menampilkan hasil prediksi
# print(forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

# Menyimpan hasil prediksi ke dalam DataFrame
forecast_df_ASP = forecast_final[['ds', 'yhat']]

07:53:24 - cmdstanpy - INFO - Chain [1] start processing
07:53:24 - cmdstanpy - INFO - Chain [1] done processing
07:53:24 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.01: 10.08%


07:53:25 - cmdstanpy - INFO - Chain [1] done processing
07:53:25 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.1: 14.58%


07:53:25 - cmdstanpy - INFO - Chain [1] done processing
07:53:26 - cmdstanpy - INFO - Chain [1] start processing


...

07:53:46 - cmdstanpy - INFO - Chain [1] done processing
07:53:47 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.01: 5.90%


07:54:06 - cmdstanpy - INFO - Chain [1] done processing
07:54:06 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.1: 48.65%

Parameter terbaik: changepoint_prior_scale=0.5, seasonality_mode=additive, seasonality_prior_scale=0.01
SMAPE terbaik: 5.56%


07:54:06 - cmdstanpy - INFO - Chain [1] done processing


In [25]:

# Menambahkan data untuk 'Forecast'
forecast_rows_SE = pd.DataFrame({
    'Date': forecast_df_ASP['ds'],
    'Tonase': forecast_df_ASP['yhat'],
    'Jenis': 'Forecast',
    'Breakdown' : 'Avg. Selling Price'
})

df_actual_sales_forecast_df_ASP = pd.DataFrame({
    'Date': df_ASP['ds'],
    'Tonase': df_ASP['y'],
    'Jenis': 'Actual',
    'Breakdown' : 'Avg. Selling Price'
})



# Menggabungkan semua DataFrame
result_df_Avg_SP = pd.concat([df_actual_sales_forecast_df_ASP,forecast_rows_SE ], ignore_index=True)
result_df_Avg_SP

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,9.300673e+05,Actual,Avg. Selling Price
1,2022-02-28,1.046655e+06,Actual,Avg. Selling Price
2,2022-03-31,1.412003e+06,Actual,Avg. Selling Price
3,2022-04-30,1.406562e+06,Actual,Avg. Selling Price
4,2022-05-31,1.279440e+06,Actual,Avg. Selling Price
...,...,...,...,...
81,2025-09-01,1.014249e+06,Forecast,Avg. Selling Price
82,2025-10-01,9.811703e+05,Forecast,Avg. Selling Price
83,2025-11-01,1.061187e+06,Forecast,Avg. Selling Price
84,2025-12-01,1.013906e+06,Forecast,Avg. Selling Price


---

In [26]:
df_ASP_Domestic = df[df['BreakDown']=='Average Domestic Price']
df_ASP_Domestic = df_ASP_Domestic[df_ASP_Domestic['tipe']=='Actual']
df_ASP_Domestic

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
553,2024-04-30,4,2024,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,786049.90,2024-04-30 05:11:01,2025-03-05 07:45:11.177
761,2023-01-31,1,2023,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,784671.54,2023-01-31 05:11:01,2025-03-05 07:45:11.050
4540,2022-05-31,5,2022,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,784225.32,2022-05-31 05:11:01,2025-03-05 07:45:11.083
6211,2023-02-28,2,2023,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,783259.54,2023-02-28 05:11:01,2025-03-05 07:45:11.067
6382,2024-01-31,1,2024,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,802335.09,2024-01-31 05:11:01,2025-03-05 07:45:11.160
6461,2022-01-31,1,2022,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,777962.82,2022-01-31 05:11:01,2025-03-05 07:45:11.020
6594,2023-03-31,3,2023,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,764892.56,2023-03-31 05:11:01,2025-03-05 07:45:11.130
7867,2023-10-31,10,2023,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,750665.39,2023-10-31 05:11:01,2025-03-05 07:45:11.143
8412,2024-02-29,2,2024,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,761575.51,2024-02-29 05:11:01,2025-03-05 07:45:11.160
8779,2022-04-30,4,2022,External Data,Average Domestic Price,IDR/Ton,None,None,Actual,918322.94,2022-04-30 05:11:01,2025-03-05 07:45:11.083


In [27]:
df_ASP_Domestic = df_ASP_Domestic.sort_values(by='Date')

df_ASP_Domestic = df_ASP_Domestic.groupby(['Date'])['amount'].sum().reset_index()
df_ASP_Domestic

,Date,amount
0,2022-01-31,777962.82
1,2022-02-28,754704.81
2,2022-03-31,782620.39
3,2022-04-30,918322.94
4,2022-05-31,784225.32
5,2022-06-30,798864.42
6,2022-07-31,1010116.57
7,2022-08-31,790729.89
8,2022-09-30,787640.80
9,2022-10-31,785098.65


In [28]:
df_ASP_Domestic.columns = ['ds','y']
df_ASP_Domestic['ds'] = pd.to_datetime(df_ASP_Domestic['ds'])

# Kombinasi parameter untuk tuning
changepoint_prior_scales = [0.01, 0.02]
seasonality_modes = ['additive']
seasonality_prior_scales = [0.01, 0.05, 0.1]

# Membuat list kombinasi parameter
param_combinations = list(itertools.product(changepoint_prior_scales, seasonality_modes, seasonality_prior_scales))

# Fungsi untuk menghitung SMAPE
def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(predicted) + np.abs(actual)))

# Menyimpan hasil tuning
best_smape = float('inf')
best_params = None

# Train dan test split
train_size = int(0.8 * len(df_ASP_Domestic))
train_data = df_ASP_Domestic[:train_size]
test_data = df_ASP_Domestic[train_size:]

# Looping untuk setiap kombinasi parameter
for changepoint_prior_scale, seasonality_mode, seasonality_prior_scale in param_combinations:
    
    # Inisialisasi model Prophet dengan parameter tuning
    model = Prophet(
        yearly_seasonality=True, 
        weekly_seasonality=True, 
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_mode=seasonality_mode,
        seasonality_prior_scale=seasonality_prior_scale,
        growth='linear'
    )
    model.add_seasonality(name='monthly', period=30, fourier_order=3)
    
    # Melatih model dengan data train
    model.fit(train_data)

    # Membuat dataframe masa depan yang mencakup periode data test
    future = model.make_future_dataframe(periods=len(test_data), freq='MS')

    # Melakukan prediksi
    forecast = model.predict(future)

    # Mengambil prediksi yang sesuai dengan data test
    forecast_test = forecast[-len(test_data):]

    # Menggabungkan data asli dengan prediksi
    test_data.loc[:, 'yhat'] = forecast_test['yhat'].values

    # Menghitung SMAPE untuk data test
    smape_value = smape(test_data['y'], test_data['yhat'])
    print(f"SMAPE dengan changepoint_prior_scale={changepoint_prior_scale}, seasonality_mode={seasonality_mode}, seasonality_prior_scale={seasonality_prior_scale}: {smape_value:.2f}%")

    # Menyimpan kombinasi parameter terbaik
    if smape_value < best_smape:
        best_smape = smape_value
        best_params = (changepoint_prior_scale, seasonality_mode, seasonality_prior_scale)

print(f"\nParameter terbaik: changepoint_prior_scale={best_params[0]}, seasonality_mode={best_params[1]}, seasonality_prior_scale={best_params[2]}")
print(f"SMAPE terbaik: {best_smape:.2f}%")

# Latih model dengan seluruh data menggunakan parameter terbaik
final_model = Prophet(
    yearly_seasonality=True, 
    weekly_seasonality=True, 
    daily_seasonality=False,
    changepoint_prior_scale=best_params[0],
    seasonality_mode=best_params[1],
    seasonality_prior_scale=best_params[2],
    growth='linear'
)
final_model.add_seasonality(name='monthly', period=30, fourier_order=3)

# Melatih model dengan seluruh data
final_model.fit(df_ASP_Domestic)

# Membuat dataframe masa depan untuk 6 bulan ke depan
future = final_model.make_future_dataframe(periods=12, freq='MS')

# Melakukan prediksi
forecast_final = final_model.predict(future)

# Menampilkan hasil prediksi
# print(forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

# Menyimpan hasil prediksi ke dalam DataFrame
forecast_ASP_Domestic = forecast_final[['ds', 'yhat']]


07:54:21 - cmdstanpy - INFO - Chain [1] start processing
07:54:21 - cmdstanpy - INFO - Chain [1] done processing
07:54:21 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.01: 7.97%


07:54:21 - cmdstanpy - INFO - Chain [1] done processing
07:54:21 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.05: 13.51%


07:54:21 - cmdstanpy - INFO - Chain [1] done processing
07:54:22 - cmdstanpy - INFO - Chain [1] start processing


...

07:54:22 - cmdstanpy - INFO - Chain [1] done processing
07:54:22 - cmdstanpy - INFO - Chain [1] start processing
07:54:23 - cmdstanpy - INFO - Chain [1] done processing


SMAPE dengan changepoint_prior_scale=0.02, seasonality_mode=additive, seasonality_prior_scale=0.05: 13.26%


07:54:23 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.02, seasonality_mode=additive, seasonality_prior_scale=0.1: 13.57%

Parameter terbaik: changepoint_prior_scale=0.01, seasonality_mode=additive, seasonality_prior_scale=0.01
SMAPE terbaik: 7.97%


07:54:23 - cmdstanpy - INFO - Chain [1] done processing


In [29]:

# Menambahkan data untuk 'Forecast'
forecast_rows_SE = pd.DataFrame({
    'Date': forecast_ASP_Domestic['ds'],
    'Tonase': forecast_ASP_Domestic['yhat'],
    'Jenis': 'Forecast',
    'Breakdown' : 'Avg. Selling Price Domestic'
})

df_actual_sales_forecast_df_ASP_Domestic = pd.DataFrame({
    'Date': df_ASP_Domestic['ds'],
    'Tonase': df_ASP_Domestic['y'],
    'Jenis': 'Actual',
    'Breakdown' : 'Avg. Selling Price Domestic'
})



# Menggabungkan semua DataFrame
result_df_Avg_SP_Dom = pd.concat([df_actual_sales_forecast_df_ASP_Domestic,forecast_rows_SE], ignore_index=True)
result_df_Avg_SP_Dom

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,777962.820000,Actual,Avg. Selling Price Domestic
1,2022-02-28,754704.810000,Actual,Avg. Selling Price Domestic
2,2022-03-31,782620.390000,Actual,Avg. Selling Price Domestic
3,2022-04-30,918322.940000,Actual,Avg. Selling Price Domestic
4,2022-05-31,784225.320000,Actual,Avg. Selling Price Domestic
...,...,...,...,...
81,2025-09-01,761665.638112,Forecast,Avg. Selling Price Domestic
82,2025-10-01,771158.701222,Forecast,Avg. Selling Price Domestic
83,2025-11-01,778092.518885,Forecast,Avg. Selling Price Domestic
84,2025-12-01,756666.060513,Forecast,Avg. Selling Price Domestic


---

In [30]:
df_ASP_Export = df[df['BreakDown']=='Average Export Price']
df_ASP_Export = df_ASP_Export[df_ASP_Export['tipe']=='Actual']
df_ASP_Export

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
421,2023-01-31,1,2023,External Data,Average Export Price,IDR/Ton,None,None,Actual,1831040.36,2023-01-31 05:11:01,2025-03-05 07:45:11.050
2159,2024-02-29,2,2024,External Data,Average Export Price,IDR/Ton,None,None,Actual,1216780.53,2024-02-29 05:11:01,2025-03-05 07:45:11.160
3795,2024-01-31,1,2024,External Data,Average Export Price,IDR/Ton,None,None,Actual,1239080.65,2024-01-31 05:11:01,2025-03-05 07:45:11.160
5364,2022-05-31,5,2022,External Data,Average Export Price,IDR/Ton,None,None,Actual,2066416.17,2022-05-31 05:11:01,2025-03-05 07:45:11.083
5962,2023-03-31,3,2023,External Data,Average Export Price,IDR/Ton,None,None,Actual,1415245.73,2023-03-31 05:11:01,2025-03-05 07:45:11.130
6254,2023-02-28,2,2023,External Data,Average Export Price,IDR/Ton,None,None,Actual,1623757.43,2023-02-28 05:11:01,2025-03-05 07:45:11.067
6697,2022-04-30,4,2022,External Data,Average Export Price,IDR/Ton,None,None,Actual,2215393.21,2022-04-30 05:11:01,2025-03-05 07:45:11.083
6810,2022-03-31,3,2022,External Data,Average Export Price,IDR/Ton,None,None,Actual,2096535.74,2022-03-31 05:11:01,2025-03-05 07:45:11.083
8064,2024-04-30,4,2024,External Data,Average Export Price,IDR/Ton,None,None,Actual,1240801.70,2024-04-30 05:11:01,2025-03-05 07:45:11.177
12186,2024-03-31,3,2024,External Data,Average Export Price,IDR/Ton,None,None,Actual,1269914.13,2024-03-31 05:11:01,2025-03-05 07:45:11.177


In [31]:
df_ASP_Export = df_ASP_Export.sort_values(by='Date')

df_ASP_Export = df_ASP_Export.groupby(['Date'])['amount'].sum().reset_index()
df_ASP_Export

,Date,amount
0,2022-01-31,1700916.45
1,2022-02-28,1770779.41
2,2022-03-31,2096535.74
3,2022-04-30,2215393.21
4,2022-05-31,2066416.17
5,2022-06-30,2072909.63
6,2022-07-31,2199046.36
7,2022-08-31,2372036.75
8,2022-09-30,1854725.70
9,2022-10-31,2431769.18


In [32]:
df_ASP_Export.columns = ['ds','y']
df_ASP_Export['ds'] = pd.to_datetime(df_ASP_Export['ds'])

# Kombinasi parameter untuk tuning
changepoint_prior_scales = [ 0.1, 0.5, 1]
seasonality_modes = ['additive']
seasonality_prior_scales = [0.01, 0.1]

# Membuat list kombinasi parameter
param_combinations = list(itertools.product(changepoint_prior_scales, seasonality_modes, seasonality_prior_scales))

# Fungsi untuk menghitung SMAPE
def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(predicted) + np.abs(actual)))

# Menyimpan hasil tuning
best_smape = float('inf')
best_params = None

# Train dan test split
train_size = int(0.8 * len(df_ASP_Export))
train_data = df_ASP_Export[:train_size]
test_data = df_ASP_Export[train_size:]

# Looping untuk setiap kombinasi parameter
for changepoint_prior_scale, seasonality_mode, seasonality_prior_scale in param_combinations:
    
    # Inisialisasi model Prophet dengan parameter tuning
    model = Prophet(
        yearly_seasonality=True, 
        weekly_seasonality=True, 
        daily_seasonality=False,
        changepoint_prior_scale=changepoint_prior_scale,
        seasonality_mode=seasonality_mode,
        seasonality_prior_scale=seasonality_prior_scale,
        growth='linear'
    )
    model.add_seasonality(name='monthly', period=30, fourier_order=3)
    
    # Melatih model dengan data train
    model.fit(train_data)

    # Membuat dataframe masa depan yang mencakup periode data test
    future = model.make_future_dataframe(periods=len(test_data), freq='MS')

    # Melakukan prediksi
    forecast = model.predict(future)

    # Mengambil prediksi yang sesuai dengan data test
    forecast_test = forecast[-len(test_data):]

    # Menggabungkan data asli dengan prediksi
    test_data.loc[:, 'yhat'] = forecast_test['yhat'].values

    # Menghitung SMAPE untuk data test
    smape_value = smape(test_data['y'], test_data['yhat'])
    print(f"SMAPE dengan changepoint_prior_scale={changepoint_prior_scale}, seasonality_mode={seasonality_mode}, seasonality_prior_scale={seasonality_prior_scale}: {smape_value:.2f}%")

    # Menyimpan kombinasi parameter terbaik
    if smape_value < best_smape:
        best_smape = smape_value
        best_params = (changepoint_prior_scale, seasonality_mode, seasonality_prior_scale)

print(f"\nParameter terbaik: changepoint_prior_scale={best_params[0]}, seasonality_mode={best_params[1]}, seasonality_prior_scale={best_params[2]}")
print(f"SMAPE terbaik: {best_smape:.2f}%")

# Latih model dengan seluruh data menggunakan parameter terbaik
final_model = Prophet(
    yearly_seasonality=True, 
    weekly_seasonality=True, 
    daily_seasonality=False,
    changepoint_prior_scale=best_params[0],
    seasonality_mode=best_params[1],
    seasonality_prior_scale=best_params[2],
    growth='linear'
)
final_model.add_seasonality(name='monthly', period=30, fourier_order=3)

# Melatih model dengan seluruh data
final_model.fit(df_ASP_Export)

# Membuat dataframe masa depan untuk 6 bulan ke depan
future = final_model.make_future_dataframe(periods=12, freq='MS')

# Melakukan prediksi
forecast_final = final_model.predict(future)

# Menampilkan hasil prediksi
# print(forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

# Menyimpan hasil prediksi ke dalam DataFrame
forecast_ASP_Export = forecast_final[['ds', 'yhat']]

07:54:36 - cmdstanpy - INFO - Chain [1] start processing
07:54:36 - cmdstanpy - INFO - Chain [1] done processing
07:54:36 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.01: 20.32%


07:54:37 - cmdstanpy - INFO - Chain [1] done processing
07:54:37 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=0.1, seasonality_mode=additive, seasonality_prior_scale=0.1: 23.50%


07:54:38 - cmdstanpy - INFO - Chain [1] done processing
07:54:38 - cmdstanpy - INFO - Chain [1] start processing


...

07:55:00 - cmdstanpy - INFO - Chain [1] done processing
07:55:00 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.01: 5.92%


07:55:21 - cmdstanpy - INFO - Chain [1] done processing
07:55:21 - cmdstanpy - INFO - Chain [1] start processing


SMAPE dengan changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.1: 59.55%

Parameter terbaik: changepoint_prior_scale=1, seasonality_mode=additive, seasonality_prior_scale=0.01
SMAPE terbaik: 5.92%


07:55:22 - cmdstanpy - INFO - Chain [1] done processing


In [33]:
# Menambahkan data untuk 'Forecast'
forecast_rows_SE = pd.DataFrame({
    'Date':forecast_ASP_Export['ds'],
    'Tonase': forecast_ASP_Export['yhat'],
    'Jenis': 'Forecast',
    'Breakdown' : 'Avg. Selling Price Export'
})

df_actual_sales_forecast_df_ASP_Export = pd.DataFrame({
    'Date': df_ASP_Export['ds'],
    'Tonase': df_ASP_Export['y'],
    'Jenis': 'Actual',
    'Breakdown' : 'Avg. Selling Price Export'
})


# Menggabungkan semua DataFrame
result_df_Avg_SP_Exp = pd.concat([df_actual_sales_forecast_df_ASP_Export,forecast_rows_SE ], ignore_index=True)
result_df_Avg_SP_Exp

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,1.700916e+06,Actual,Avg. Selling Price Export
1,2022-02-28,1.770779e+06,Actual,Avg. Selling Price Export
2,2022-03-31,2.096536e+06,Actual,Avg. Selling Price Export
3,2022-04-30,2.215393e+06,Actual,Avg. Selling Price Export
4,2022-05-31,2.066416e+06,Actual,Avg. Selling Price Export
...,...,...,...,...
81,2025-09-01,1.148036e+06,Forecast,Avg. Selling Price Export
82,2025-10-01,1.054503e+06,Forecast,Avg. Selling Price Export
83,2025-11-01,1.156312e+06,Forecast,Avg. Selling Price Export
84,2025-12-01,1.142294e+06,Forecast,Avg. Selling Price Export


---

In [34]:
df_Final = pd.concat((result_df_Sales, result_df_SD, result_df_Sales_Export, result_df_Avg_SP, result_df_Avg_SP_Dom, result_df_Avg_SP_Exp), axis = 0)
df_Final

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,2.091660e+06,Actual,Total Sales
1,2022-02-28,2.056356e+06,Actual,Total Sales
2,2022-03-31,2.818668e+06,Actual,Total Sales
3,2022-04-30,2.728947e+06,Actual,Total Sales
4,2022-05-31,2.654839e+06,Actual,Total Sales
...,...,...,...,...
81,2025-09-01,1.148036e+06,Forecast,Avg. Selling Price Export
82,2025-10-01,1.054503e+06,Forecast,Avg. Selling Price Export
83,2025-11-01,1.156312e+06,Forecast,Avg. Selling Price Export
84,2025-12-01,1.142294e+06,Forecast,Avg. Selling Price Export


In [35]:
df_Final = df_Final.reset_index(drop=True)
print(df_Final)


          Date        Tonase     Jenis                  Breakdown
0   2022-01-31  2.091660e+06    Actual                Total Sales
1   2022-02-28  2.056356e+06    Actual                Total Sales
2   2022-03-31  2.818668e+06    Actual                Total Sales
3   2022-04-30  2.728947e+06    Actual                Total Sales
4   2022-05-31  2.654839e+06    Actual                Total Sales
..         ...           ...       ...                        ...
509 2025-09-01  1.148036e+06  Forecast  Avg. Selling Price Export
510 2025-10-01  1.054503e+06  Forecast  Avg. Selling Price Export
511 2025-11-01  1.156312e+06  Forecast  Avg. Selling Price Export
512 2025-12-01  1.142294e+06  Forecast  Avg. Selling Price Export
513 2026-01-01  1.008743e+06  Forecast  Avg. Selling Price Export

[514 rows x 4 columns]


---

In [63]:
# DF Plan
df_plan = df[df['tipe']=='RKAP'].copy()
df_plan

,Date,bulan,tahun,Indicator,BreakDown,Unit,Country,Source,tipe,amount,last_updated,Updated_Date
1,2023-01-31,1,2023,Operating Expenses,Fuel Usage of Marketing,IDR,None,None,RKAP,2.840146e+09,2023-01-31 05:11:01,2025-03-05 07:45:11.143
7,2023-01-31,1,2023,Operating Expenses,Other Selling Expenses of Marketing,IDR,None,None,RKAP,3.358052e+10,2023-01-31 05:11:01,2025-03-05 07:45:11.050
13,2023-01-31,1,2023,Cash Conversion Cycle,Cash Conversion Cycle,Days,None,None,RKAP,0.000000e+00,2023-01-31 05:11:01,2025-03-05 07:45:11.050
19,2023-01-31,1,2023,Cost of Goods Sold,"Salaries, Wages, and Employee Benefits",IDR,None,None,RKAP,7.293799e+10,2023-01-31 05:11:01,2025-03-05 07:45:11.050
25,2023-01-31,1,2023,Current Assets,Total Current Assets,IDR,None,None,RKAP,2.305630e+13,2023-01-31 05:11:01,2025-03-05 07:45:11.050
...,...,...,...,...,...,...,...,...,...,...,...,...
49383,2024-08-31,8,2024,Financing Activities Details,Payment from stockholders loan,IDR,None,None,RKAP,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49389,2024-08-31,8,2024,Financing Activities Details,Payment to stockholders loan,IDR,None,None,RKAP,0.000000e+00,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49395,2024-08-31,8,2024,Non Current Assets,Others of Non Current Assets,IDR,None,None,RKAP,8.036372e+12,2024-08-31 05:11:01,2025-03-05 07:45:11.177
49401,2024-08-31,8,2024,Operating Expenses,Business Travel and Education,IDR,None,None,RKAP,1.189166e+10,2024-08-31 05:11:01,2025-03-05 07:45:11.177


In [64]:
df_plan_sales = df_plan[df_plan['BreakDown'].isin(['Total Domestic Sales of Coal','Total Export Sales of Coal'])].copy()
df_plan_sales = df_plan_sales.groupby(['Date'])['amount'].sum().reset_index()
df_plan_sales

,Date,amount
0,2022-01-31,2746000.00
1,2022-02-28,2791200.00
2,2022-03-31,2988900.00
3,2022-04-30,2932200.00
4,2022-05-31,3100500.00
5,2022-06-30,3018300.00
6,2022-07-31,3198600.00
7,2022-08-31,3231600.00
8,2022-09-30,3187800.00
9,2022-10-31,3323900.00


In [66]:
df_plan_sales_domestik = df_plan[df_plan['BreakDown'] == 'Total Domestic Sales of Coal'].groupby(['Date'])['amount'].sum().reset_index()
df_plan_sales_export = df_plan[df_plan['BreakDown'] == 'Total Export Sales of Coal'].groupby(['Date'])['amount'].sum().reset_index()
df_plan_ASP_Export = df_plan[df_plan['BreakDown'] == 'Average Export Price'].groupby(['Date'])['amount'].sum().reset_index()
df_plan_ASP = df_plan[df_plan['BreakDown'] == 'Average Market Price IDR'].groupby(['Date'])['amount'].sum().reset_index()
df_plan_ASP_Domestic = df_plan[df_plan['BreakDown'] == 'Average Domestic Price'].groupby(['Date'])['amount'].sum().reset_index()


In [67]:
from datetime import datetime
# Menambahkan data untuk 'Forecast'
df_plan_sales = pd.DataFrame({
    'Date':df_plan_sales['Date'],
    'Tonase': df_plan_sales['amount'],
    'Jenis': 'Plan',
    'Breakdown' : 'Total Sales'
})

df_plan_sales_domestik = pd.DataFrame({
    'Date': df_plan_sales_domestik['Date'],
    'Tonase': df_plan_sales_domestik['amount'],
    'Jenis': 'Plan',
    'Breakdown' : 'Total Sales Domestik'
})
df_plan_sales_export = pd.DataFrame({
    'Date': df_plan_sales_export['Date'],
    'Tonase': df_plan_sales_export['amount'],
    'Jenis': 'Plan',
    'Breakdown' : 'Total Sales Ekspor'
})
df_plan_ASP = pd.DataFrame({
    'Date': df_plan_ASP['Date'],
    'Tonase': df_plan_ASP['amount'],
    'Jenis': 'Plan',
    'Breakdown' : 'Avg. Selling Price'
})
df_plan_ASP_Export = pd.DataFrame({
    'Date': df_plan_ASP_Export['Date'],
    'Tonase': df_plan_ASP_Export['amount'],
    'Jenis': 'Plan',
    'Breakdown' : 'Avg. Selling Price Export'
})
df_plan_ASP_Domestic = pd.DataFrame({
    'Date': df_plan_ASP_Domestic['Date'],
    'Tonase': df_plan_ASP_Domestic['amount'],
    'Jenis': 'Plan',
    'Breakdown' : 'Avg. Selling Price Domestic'
})
new_row = pd.DataFrame({
    'Date': [datetime.now()],  # Menggunakan datetime sekarang
    'Tonase': [0],  # Tonase diisi 0
    'Jenis': ['Actual'],  # Jenis diisi "Actual"
    'Breakdown': ['Last Update']  # Breakdown diisi "Last Update"
})

# Menggabungkan semua DataFrame
df_All = pd.concat([df_plan_sales, df_plan_sales_domestik, df_plan_sales_export, df_plan_ASP, df_plan_ASP_Domestic, df_plan_ASP_Export, df_Final, new_row ], ignore_index=True)
df_All

,Date,Tonase,Jenis,Breakdown
0,2022-01-31,2.746000e+06,Plan,Total Sales
1,2022-02-28,2.791200e+06,Plan,Total Sales
2,2022-03-31,2.988900e+06,Plan,Total Sales
3,2022-04-30,2.932200e+06,Plan,Total Sales
4,2022-05-31,3.100500e+06,Plan,Total Sales
...,...,...,...,...
732,2025-10-01 00:00:00,1.054503e+06,Forecast,Avg. Selling Price Export
733,2025-11-01 00:00:00,1.156312e+06,Forecast,Avg. Selling Price Export
734,2025-12-01 00:00:00,1.142294e+06,Forecast,Avg. Selling Price Export
735,2026-01-01 00:00:00,1.008743e+06,Forecast,Avg. Selling Price Export


In [68]:
df_All['Date'] = pd.to_datetime(df_All['Date'])
df_All['Tonase'] = round(df_All['Tonase'], 2)

In [69]:
df_All.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 737 entries, 0 to 736
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       737 non-null    datetime64[ns]
 1   Tonase     737 non-null    float64       
 2   Jenis      737 non-null    object        
 3   Breakdown  737 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 23.2+ KB


In [52]:
import pyodbc
from dotenv import load_dotenv
import os
import pandas as pd

# Muat variabel lingkungan dari file .env
# load_dotenv()
# server = os.environ['SERVER']
# database = os.environ['DATABASE']
# username = os.environ['USERNAME_1']
# password = os.environ['PASSWORD']

# SERVER=10.3.4.139,1433
# DATABASE=dwh_prod
# USERNAME_1=developerptba
# PASSWORD=LnfPYVFW1K2TAKf
# Hardcode nilai koneksi SQL Server
server = '10.3.4.139,1433'
database = 'dwh_prod'
username = 'developerptba'
password = 'LnfPYVFW1K2TAKf'

# Koneksi ke SQL Server
try:
    # conn = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
    conn = pyodbc.connect(f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}')
    print("Connection successful")
    
    cursor = conn.cursor() 

    # Truncate tabel
    truncate_query = "TRUNCATE TABLE dwh.DM_Sales_Forecasting"
    cursor.execute(truncate_query)
    print("Table truncated successfully.")

    # Setting fast_executemany
    cursor.fast_executemany = True

    insert_query = """
    INSERT INTO dwh.DM_Sales_Forecasting 
    ([Date], [Tonase], [Jenis], [Breakdown]) 
    VALUES (?, ?, ?, ?)
    """

    # Siapkan data yang akan di-insert
    rows_to_insert = [row for row in df_All.itertuples(index=False)]
    print(f"{len(rows_to_insert)} rows will be inserted into the database.")

    # Nonaktifkan autocommit
    conn.autocommit = False

    # Menggunakan try-except saat insert
    for row in rows_to_insert:
        cursor.execute(insert_query, row)
        
    # Commit perubahan jika semua insert berhasil
    conn.commit()
    print("Data successfully inserted.")

except Exception as e:
    print(f"An error occurred: {e}")
    if conn:
        conn.rollback()  # Rollback jika terjadi error

finally:
    cursor.close()
    conn.close()  # Pastikan koneksi selalu ditutup
    print("Connection closed.")


Connection successful
Table truncated successfully.
736 rows will be inserted into the database.
Data successfully inserted.
Connection closed.


In [70]:
# print("Sedang Import ke DM_forecast_prod_v1")

import pyodbc
from dotenv import load_dotenv
import os
import pandas as pd

# Koneksi ke SQL Server
server = '10.3.4.139,1433'
database = 'dwh_prod'
username = 'developerptba'
password = 'LnfPYVFW1K2TAKf'

# Inisialisasi koneksi ke SQL Server
try:
    conn = pyodbc.connect(
        f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
    )
    print("Koneksi ke database berhasil.")
except Exception as e:
    print(f"Error saat mencoba koneksi ke database: {e}")
    exit()

cursor = conn.cursor()

# Proses truncate tabel
truncate_query = "TRUNCATE TABLE dwh.DM_Sales_Forecasting"
try:
    cursor.execute(truncate_query)
    conn.commit()
    print("Tabel berhasil di-truncate.")
except Exception as e:
    print(f"Error saat truncate tabel: {e}")
    conn.close()
    exit()


# Pastikan `Df_Final` sudah didefinisikan
if 'df_All' not in globals():
    print("Error: DataFrame `df_All` tidak ditemukan.")
    conn.close()
    exit()

# Menyiapkan query untuk insert
insert_query = """
    INSERT INTO dwh.DM_Sales_Forecasting 
    ([Date], [Tonase], [Jenis], [Breakdown]) 
    VALUES (?, ?, ?, ?)
"""

rows_to_insert = [tuple(row) for row in df_All.itertuples(index=False)]

# Proses insert data
cursor.fast_executemany = True
try:
    cursor.executemany(insert_query, rows_to_insert)
    conn.commit()
    print(f"Berhasil Insert {len(rows_to_insert)} baris data.")
except Exception as e:
    conn.rollback()
    print(f"Error saat menginsert data: {e}")

# Menutup koneksi
cursor.close()
conn.close()
print("Forecasting Done")


Koneksi ke database berhasil.
Tabel berhasil di-truncate.
Berhasil Insert 737 baris data.
Forecasting Done


In [25]:
result_df.to_csv('df_sales.csv')